<a href="https://colab.research.google.com/github/Kwasi-Scientist/Algorithms/blob/main/ChurnPrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas numpy matplotlib seaborn scikit-learn



In [59]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from IPython.display import display
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
df = pd.read_csv("/content/drive/My Drive/Colab Notebooks/MLBookCamp/Churn/Customer-Churn.csv")

In [5]:
len(df)

7043

In [6]:
df.head().T

,0,1,2,3,4
customerID,7590-VHVEG,5575-GNVDE,3668-QPYBK,7795-CFOCW,9237-HQITU
gender,Female,Male,Male,Male,Female
SeniorCitizen,0,0,0,0,0
Partner,Yes,No,No,No,No
Dependents,No,No,No,No,No
tenure,1,34,2,45,2
PhoneService,No,Yes,Yes,No,Yes
MultipleLines,No phone service,No,No,No phone service,No
InternetService,DSL,DSL,DSL,DSL,Fiber optic
OnlineSecurity,No,Yes,Yes,Yes,No


In [7]:
df.dtypes

,0
customerID,object
gender,object
SeniorCitizen,int64
Partner,object
Dependents,object
tenure,int64
PhoneService,object
MultipleLines,object
InternetService,object
OnlineSecurity,object


In [8]:
df.isnull().sum()

,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


In [9]:
#change total charges column to numeric

total_charges = pd.to_numeric(df.TotalCharges, errors = 'coerce')
df[total_charges.isnull()][['TotalCharges']]

,TotalCharges
488,
753,
936,
1082,
1340,
3331,
3826,
4380,
5218,
6670,


In [10]:
#there are empty spaces in total charges, so fill them with zeros

df.TotalCharges = pd.to_numeric(df.TotalCharges, errors = 'coerce')
df.TotalCharges = df.TotalCharges.fillna(0)

In [11]:
#adjust naming convention of the columns
#make all columns lowercase and change spaces to underscores
df.columns = df.columns.str.lower().str.replace(' ', '_')
string_columns = list(df.dtypes[df.dtypes == 'object'].index)

for col in string_columns:
  df[col] = df[col].str.lower().str.replace(' ', '_')

In [12]:
#look at target variable `churn` and convert the categorical yes and no to binary numbners 0 (no) and 1 (yes)

df.churn = (df.churn == 'yes').astype(int)

In [13]:
df_train_full, df_test_full = train_test_split(df, test_size =0.2, random_state=1)

In [14]:
df_train_full.head()

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
1814,5442-pptjy,male,0,yes,yes,12,yes,no,no,no_internet_service,...,no_internet_service,no_internet_service,no_internet_service,no_internet_service,two_year,no,mailed_check,19.70,258.35,0
5946,6261-rcvns,female,0,no,no,42,yes,no,dsl,yes,...,yes,yes,no,yes,one_year,no,credit_card_(automatic),73.90,3160.55,1
3881,2176-osjuv,male,0,yes,no,71,yes,yes,dsl,yes,...,no,yes,no,no,two_year,no,bank_transfer_(automatic),65.15,4681.75,0
2389,6161-erdgd,male,0,yes,yes,71,yes,yes,dsl,yes,...,yes,yes,yes,yes,one_year,no,electronic_check,85.45,6300.85,0
3676,2364-ufrom,male,0,no,no,30,yes,no,dsl,yes,...,no,yes,yes,no,one_year,no,electronic_check,70.40,2044.75,0


In [15]:
#split df_train_full into train and validation sets

df_train, df_val = train_test_split(df_train_full, test_size=0.33, random_state=11)

In [16]:
#define ytrain as df_train.churn.values
#define y_val as df_val.churn.values
y_train = df_train.churn.values
y_val = df_val.churn.values

#delet churn values from df_train and df_val

del df_train['churn']
del df_val['churn']

Explratory Data Analysis

In [17]:
#check for missing values
df_train_full.isnull().sum()

,0
customerid,0
gender,0
seniorcitizen,0
partner,0
dependents,0
tenure,0
phoneservice,0
multiplelines,0
internetservice,0
onlinesecurity,0


In [18]:
#check the distribution of the values in the target variable
df_train_full.churn.value_counts()

,count
churn,
0,4113
1,1521


In [19]:
#check the proportion of churned users among all customers

1521/(1521+4113)

#this proportion of churned users is the propbablity that a customer will churn i.e. the churn rate

0.26996805111821087

In [20]:
#we can also calculate churn rate using the mean
global_mean = df_train_full.churn.mean()

In [21]:
round(global_mean,3)

np.float64(0.27)

In [22]:
  #the dataset is imbalanced because there are 3 times as many people who did not chrun compared to those who did
  #the nonchurn class dominates the churn class

In [23]:
#handle the numerical and categorical values separetly
categorical = ['gender', 'seniorcitizen', 'partner', 'dependents',
               'phoneservice', 'multiplelines', 'internetservice',
               'onlinesecurity', 'onlinebackup', 'deviceprotection',
               'techsupport', 'streamingtv', 'streamingmovies',
               'contract', 'paperlessbilling', 'paymentmethod']
numerical = ['tenure', 'monthlycharges', 'totalcharges']

In [24]:
#check the unique values each categorical has
df_train_full[categorical].nunique()

,0
gender,2
seniorcitizen,2
partner,2
dependents,2
phoneservice,2
multiplelines,3
internetservice,3
onlinesecurity,3
onlinebackup,3
deviceprotection,3


Feature Importance Analysis

In [25]:
#compute the churn rate for each categorical group and then compare it to the global chrun rate
#the global churn rate is the churn rate of all observations at once
#If the delta of the churn rates is small the categorical value is not important
#if the delta is large, the categorical value is important

#calculate the churn rate for `gender` where `gender` == `female`

female_mean = df_train_full[df_train_full.gender=='female'].churn.mean()

#do the same for male
male_mean = df_train_full[df_train_full.gender=='male'].churn.mean()

In [26]:
female_mean, male_mean, global_mean

(np.float64(0.27682403433476394),
 np.float64(0.2632135306553911),
 np.float64(0.26996805111821087))

In [27]:
#the female, male, and global churn rate are all similar, i.e the delta is small so the `gender` feature is not significant

In [28]:
#looking at `partner` feature

partner_yes = df_train_full[df_train_full.partner=='yes'].churn.mean()
partner_no = df_train_full[df_train_full.partner=='no'].churn.mean()

In [29]:
partner_yes, partner_no, global_mean

(np.float64(0.20503330866025166),
 np.float64(0.3298090040927694),
 np.float64(0.26996805111821087))

In [30]:
#partner_yes is signifcantly lower than parnter_no and glbalmean so it is significant

Risk Ratio

In [31]:
#the risk ratio is the ratio between probabilities in different groups
#Where risk is the potential for having an effect (i.e. churn)
#risk = group rate/global rate
#if the difference between the group rate and the global rate is small the risk ratio is close to 1
#a group with a risk clsoe to 1 is not risky
#a risk ratio lower than 1 indicates very low risk i.e a risk of 0.5 means the group is tiwce as likely not to churn in our case
#if the difference betweel the group rate and the global rate is large the risk ratio is larger than 1


In [39]:
#calculate churn rate for each categorical value


#calc global average churn
global_mean = df_train_full.churn.mean()

#iter through categoricals
for col in categorical:

  ##compute average churn for categorical
  df_group = df_train_full.groupby(by=col).churn.agg(['mean'])

  #calc the difference between the group chrun rate and global churn rate and save to new column
  df_group['diff'] = df_group['mean'] - global_mean

  #calculate risk of churnning
  df_group['risk'] = df_group['mean'] / global_mean

  display(df_group)


,mean,diff,risk
gender,,,
female,0.276824,0.006856,1.025396
male,0.263214,-0.006755,0.974980


,mean,diff,risk
seniorcitizen,,,
0,0.242270,-0.027698,0.897403
1,0.413377,0.143409,1.531208


,mean,diff,risk
partner,,,
no,0.329809,0.059841,1.221659
yes,0.205033,-0.064935,0.759472


,mean,diff,risk
dependents,,,
no,0.313760,0.043792,1.162212
yes,0.165666,-0.104302,0.613651


,mean,diff,risk
phoneservice,,,
no,0.241316,-0.028652,0.893870
yes,0.273049,0.003081,1.011412


...

,mean,diff,risk
streamingtv,,,
no,0.342832,0.072864,1.269897
no_internet_service,0.077805,-0.192163,0.288201
yes,0.302723,0.032755,1.121328


,mean,diff,risk
streamingmovies,,,
no,0.338906,0.068938,1.255358
no_internet_service,0.077805,-0.192163,0.288201
yes,0.307273,0.037305,1.138182


,mean,diff,risk
contract,,,
month-to-month,0.431701,0.161733,1.599082
one_year,0.120573,-0.149395,0.446621
two_year,0.028274,-0.241694,0.104730


,mean,diff,risk
paperlessbilling,,,
no,0.172071,-0.097897,0.637375
yes,0.338151,0.068183,1.252560


,mean,diff,risk
paymentmethod,,,
bank_transfer_(automatic),0.168171,-0.101797,0.622928
credit_card_(automatic),0.164339,-0.105630,0.608733
electronic_check,0.455890,0.185922,1.688682
mailed_check,0.193870,-0.076098,0.718121


In [31]:
#remember finding the risk ratio can help us determine which features will be useful in our mode

Mutual Information

In [41]:
#Mutual information allows us to determine how much information we learn about one variable if we learn the value of the other variable
#use it to measure mutual dependancy between two variables
#higher values of mutual infomation mena a higher degree of dependence
#if the mutual information between the categorical variable and the target is high then the categorical variable will be very useful for predicting the target
#use the mutual_info_score() function  from `metrics` package in `sklearn`
#does not work when a feature is numerical

In [43]:
#calculate mutual information of categoricals
def calculate_mi(series): #takes one param a series wich is the column of categoricals from the df
  return mutual_info_score(series, df_train_full.churn)


df_mi = df_train_full[categorical].apply(calculate_mi) #apply  function to each categorical column of the df
df_mi = df_mi.sort_values(ascending=False).to_frame(name= 'MI') #sort values of the results
df_mi

#the most useful categoricals appear first in the df as we sorted by ascending = False

,MI
contract,0.098320
onlinesecurity,0.063085
techsupport,0.061032
internetservice,0.055868
onlinebackup,0.046923
deviceprotection,0.043453
paymentmethod,0.043210
streamingtv,0.031853
streamingmovies,0.031581
paperlessbilling,0.017589


Correlation Coefficient

In [44]:
#Pearsons correlation coefficient
# has a values from -1 to 1
#measures the dependency between variables and target just like mutual information
#a positve correlation means that when one variable goes up the other variable also tends to go up
  #in a binary situation we with a positive correlation we would see 1s more often than 0s
#zero correlation means there is no releationship and the variables are independant
#negative correlation means that when one variable goes up the other tends to go down


In [45]:
#caluclate the Pearsons correlation coefficient using the `.corrwith()` fuction
df_train_full[numerical].corrwith(df_train_full.churn)

,0
tenure,-0.351885
monthlycharges,0.196805
totalcharges,-0.196353


Feature Engineering

In [47]:
#One hot encoding of categorical variables using `DictVectorizer` from sklearn

train_dict = df_train[categorical + numerical].to_dict(orient='records')
train_dict

[{'gender': 'male',
  'seniorcitizen': 0,
  'partner': 'yes',
  'dependents': 'no',
  'phoneservice': 'yes',
  'multiplelines': 'no',
  'internetservice': 'dsl',
  'onlinesecurity': 'yes',
  'onlinebackup': 'yes',
  'deviceprotection': 'yes',
  'techsupport': 'yes',
  'streamingtv': 'yes',
  'streamingmovies': 'yes',
  'contract': 'two_year',
  'paperlessbilling': 'yes',
  'paymentmethod': 'bank_transfer_(automatic)',
  'tenure': 71,
  'monthlycharges': 86.1,
  'totalcharges': 6045.9},
 {'gender': 'female',
  'seniorcitizen': 1,
  'partner': 'yes',
  'dependents': 'no',
  'phoneservice': 'yes',
  'multiplelines': 'yes',
  'internetservice': 'fiber_optic',
  'onlinesecurity': 'no',
  'onlinebackup': 'no',
  'deviceprotection': 'yes',
  'techsupport': 'no',
  'streamingtv': 'yes',
  'streamingmovies': 'yes',
  'contract': 'one_year',
  'paperlessbilling': 'yes',
  'paymentmethod': 'credit_card_(automatic)',
  'tenure': 60,
  'monthlycharges': 100.5,
  'totalcharges': 6029.0},
 {'gender':

In [49]:
#one hot encoding
dv = DictVectorizer(sparse=False)
dv.fit(train_dict)

DictVectorizer(sparse=False)

In [50]:
#convert the dit to a matrix using `transform` method
x_train = dv.transform(train_dict)

In [51]:
x_train[0]

array([0.0000e+00, 0.0000e+00, 1.0000e+00, 1.0000e+00, 0.0000e+00,
       0.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00, 1.0000e+00,
       1.0000e+00, 0.0000e+00, 0.0000e+00, 8.6100e+01, 1.0000e+00,
       0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00,
       0.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00, 1.0000e+00,
       0.0000e+00, 1.0000e+00, 1.0000e+00, 0.0000e+00, 0.0000e+00,
       0.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00, 0.0000e+00,
       0.0000e+00, 1.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00,
       0.0000e+00, 0.0000e+00, 1.0000e+00, 7.1000e+01, 6.0459e+03])

In [56]:
dv.get_feature_names_out()

array(['contract=month-to-month', 'contract=one_year',
       'contract=two_year', 'dependents=no', 'dependents=yes',
       'deviceprotection=no', 'deviceprotection=no_internet_service',
       'deviceprotection=yes', 'gender=female', 'gender=male',
       'internetservice=dsl', 'internetservice=fiber_optic',
       'internetservice=no', 'monthlycharges', 'multiplelines=no',
       'multiplelines=no_phone_service', 'multiplelines=yes',
       'onlinebackup=no', 'onlinebackup=no_internet_service',
       'onlinebackup=yes', 'onlinesecurity=no',
       'onlinesecurity=no_internet_service', 'onlinesecurity=yes',
       'paperlessbilling=no', 'paperlessbilling=yes', 'partner=no',
       'partner=yes', 'paymentmethod=bank_transfer_(automatic)',
       'paymentmethod=credit_card_(automatic)',
       'paymentmethod=electronic_check', 'paymentmethod=mailed_check',
       'phoneservice=no', 'phoneservice=yes', 'seniorcitizen',
       'streamingmovies=no', 'streamingmovies=no_internet_service',

Train Logistic Regrssion Model

In [57]:
#logistic regression yeilds a probability that the observation is positive
#must make sure the prediction of the model is between 0 and 1 using the sigmoid function

In [60]:
model = LogisticRegression(solver ="liblinear", random_state=1)
model.fit(x_train, y_train)

LogisticRegression(random_state=1, solver='liblinear')

In [62]:
#apply model to validation set

#create validation set
val_dict = df_val[categorical + numerical].to_dict(orient='records') #perform one hot encoding
x_val = dv.transform(val_dict)  #transform into a matrix

In [63]:
#predict using the `predict_proba()` method

y_pred = model.predict_proba(x_val)

In [64]:
#two columns are produced
#the first column is the probability that the observation belongs to the negative class ( customers will NOT churn)
#the second column is the probability that the observations belong to the positive class (customers will chrun)
#we want to take only one of the columns, the positive class

y_pred

array([[0.76508784, 0.23491216],
       [0.73113015, 0.26886985],
       [0.68054704, 0.31945296],
       ...,
       [0.94274614, 0.05725386],
       [0.38476895, 0.61523105],
       [0.93872763, 0.06127237]])

In [65]:
#slice the prediction to keep only the second column, positive class column
y_pred = model.predict_proba(x_val)[:,1]

In [66]:
y_pred

#this is a "soft" prediction

array([0.23491216, 0.26886985, 0.31945296, ..., 0.05725386, 0.61523105,
       0.06127237])

In [68]:
#get a "hard" prediction by applying a cutoff of 0.5 so we can make the predicitons binary
#i.e simply use the following expression

churn = y_pred >= 0.5


#if the soft prediction is >= 0.5 then it is True, else False

In [69]:
#Now determine the quality of the prediction
#determine the quality by calculating the accuracy of the model by comparing the hard prediction to the actual churn values (y_val)

(y_val == churn).mean()
#remember y_val is our target data



#there is an 80% accuracy which is decent

np.float64(0.8016129032258065)

Model Interpretation

In [70]:
#the weights are are stored in `model.coef_[0]` and the intercept is stored in `model.intercept_[0]`
#get the weights

dict(zip(dv.get_feature_names_out(), model.coef_[0].round(3)))

{'contract=month-to-month': np.float64(0.563),
 'contract=one_year': np.float64(-0.086),
 'contract=two_year': np.float64(-0.599),
 'dependents=no': np.float64(-0.03),
 'dependents=yes': np.float64(-0.092),
 'deviceprotection=no': np.float64(0.1),
 'deviceprotection=no_internet_service': np.float64(-0.116),
 'deviceprotection=yes': np.float64(-0.106),
 'gender=female': np.float64(-0.027),
 'gender=male': np.float64(-0.095),
 'internetservice=dsl': np.float64(-0.323),
 'internetservice=fiber_optic': np.float64(0.317),
 'internetservice=no': np.float64(-0.116),
 'monthlycharges': np.float64(0.001),
 'multiplelines=no': np.float64(-0.168),
 'multiplelines=no_phone_service': np.float64(0.127),
 'multiplelines=yes': np.float64(-0.081),
 'onlinebackup=no': np.float64(0.136),
 'onlinebackup=no_internet_service': np.float64(-0.116),
 'onlinebackup=yes': np.float64(-0.142),
 'onlinesecurity=no': np.float64(0.258),
 'onlinesecurity=no_internet_service': np.float64(-0.116),
 'onlinesecurity=yes':

In [73]:
#take a small subset of features and retrain to understand the model
small_subset= ["contract", "tenure","totalcharges"]
train_dict_small = df_train[small_subset].to_dict(orient='records')
dv_small = DictVectorizer(sparse=False)
dv_small.fit(train_dict_small)

DictVectorizer(sparse=False)

In [74]:
x_small_train = dv_small.transform(train_dict_small)

In [75]:
dv_small.get_feature_names_out()

array(['contract=month-to-month', 'contract=one_year',
       'contract=two_year', 'tenure', 'totalcharges'], dtype=object)

In [77]:
model_small = LogisticRegression(solver='liblinear', random_state=1)
model_small.fit(x_small_train, y_train)

LogisticRegression(random_state=1, solver='liblinear')

In [79]:
model_small.intercept_[0]

np.float64(-0.6387618613273348)

In [84]:
dict(zip(dv_small.get_feature_names_out(), model_small.coef_[0].round(3)))

{'contract=month-to-month': np.float64(0.91),
 'contract=one_year': np.float64(-0.144),
 'contract=two_year': np.float64(-1.404),
 'tenure': np.float64(-0.097),
 'totalcharges': np.float64(0.001)}